<a href="https://colab.research.google.com/github/sun-mengwei/dtb-colab-experiments/blob/elliptic-pde/dtb_elliptic_single_mode_dimensions_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DTB--Ritz single-mode Poisson in 2D, 5D, and 10D

On $[-1,1]^d$, the manufactured reference is
$$u_*(x)=\prod_{j=1}^d\cos(\pi x_j/2),\qquad f=(d\pi^2/4)u_*.$$
The exact boundary factor is tested as dimension grows. Here 'time' means outer DTB iteration: snapshots compare the fixed reference solution with the current DTB diagonal slice.

In [ ]:
# Clone the requested branch when running in a fresh Colab session.
import os, subprocess
from pathlib import Path
if Path('DTB_elliptic_utils.py').exists():
    ROOT = Path.cwd()
else:
    ROOT = Path('/content/dtb-colab-experiments')
    if not ROOT.exists():
        subprocess.run(['git', 'clone', '-b', 'elliptic-pde', 'https://github.com/sun-mengwei/dtb-colab-experiments.git', str(ROOT)], check=True)
    os.chdir(ROOT)
print('repository:', ROOT)

In [ ]:
# Shared DTB mechanics live in one concise utility module.
import math
import matplotlib.pyplot as plt
import numpy as np
import torch
from DTB_elliptic_utils import (MLP, armijo_update, direct_ritz_energy, envelope_gradient, expand_direction, flatten_parameters, make_box_trial, matrix_ritz_energy, predict_tangent, relative_l2_error, sample_box, sample_box_boundary, select_parameter_indices, solve_tangent_ritz, summarize_history)

torch.set_default_dtype(torch.float64)
DEVICE = torch.device('cpu')
CASES = [
    dict(d=2, train=512, valid=2048, outer=30),
    dict(d=5, train=512, valid=2048, outer=20),
    dict(d=10, train=768, valid=2048, outer=12),
]
WIDTH, DEPTH, TANGENT_DIM = 16, 2, 32
RIDGE_RELATIVE, INITIAL_STEP = 1e-4, 1.0

## Runner

Every outer iteration rebuilds the selected tangent basis, solves the inner Ritz system, records only $L^2$, $F(\theta)$, inner residual, and $\|\alpha\|_2$, then takes an envelope-gradient step. The plotted $F(\theta)$ is the unregularized empirical Ritz energy; the small ridge only stabilizes the inner linear solve.

In [ ]:
results = {}
for cfg in CASES:
    d, outer_steps = cfg['d'], cfg['outer']
    torch.manual_seed(100 + d)
    model = MLP(d, width=WIDTH, depth=DEPTH).to(DEVICE)
    theta, spec = flatten_parameters(model)
    theta = theta.to(DEVICE)
    trial = make_box_trial(model, spec, normalize_boundary=True)
    indices = select_parameter_indices(theta.numel(), min(TANGENT_DIM, theta.numel()), 700 + d)
    volume = 2.0 ** d

    def exact(point):
        return torch.prod(torch.cos(0.5 * math.pi * point), dim=-1)

    x_train = sample_box(cfg['train'], d, 10 + d, device=DEVICE)
    x_valid = sample_box(cfg['valid'], d, 20 + d, device=DEVICE)
    f_train = (d * math.pi**2 / 4.0) * exact(x_train)
    snapshot_steps = {0, outer_steps // 2, outer_steps}
    diagonal = torch.linspace(-1, 1, 301)[:, None].repeat(1, d)
    history, snapshots = [], {}

    for step in range(outer_steps + 1):
        solution, stiffness, load = solve_tangent_ritz(
            trial, theta, indices, x_train, f_train, volume,
            ridge_relative=RIDGE_RELATIVE,
        )
        direction = expand_direction(solution.alpha, indices, theta.numel())
        F_theta = float(matrix_ritz_energy(stiffness, load, solution.alpha))
        row = dict(
            step=step, F_theta=F_theta,
            relative_l2=relative_l2_error(trial, theta, direction, x_valid, exact),
            inner_residual=solution.relative_residual,
            alpha_norm=float(torch.linalg.norm(solution.alpha)),
        )
        history.append(row)
        if step in snapshot_steps:
            snapshots[step] = predict_tangent(trial, theta, direction, diagonal).detach().cpu().numpy()
        if step == outer_steps:
            break

        # Alpha is fixed in this outer derivative (envelope step).
        gradient = envelope_gradient(trial, theta, direction, x_train, f_train, volume)
        fixed_objective = lambda th: direct_ritz_energy(trial, th, direction.detach(), x_train, f_train, volume)
        theta, _ = armijo_update(fixed_objective, theta, gradient, initial_step=INITIAL_STEP)

    boundary = sample_box_boundary(64, d, 900 + d)
    boundary_max = float(predict_tangent(trial, theta, direction, boundary).abs().max())
    results[d] = dict(history=history, snapshots=snapshots, diagonal=diagonal[:, 0].numpy(), boundary_max=boundary_max)
    print(f'\n--- d={d}: max |u_DTB| on sampled boundary = {boundary_max:.3e} ---')
    summarize_history([history[0], history[len(history)//2], history[-1]])

## Reference-versus-DTB snapshots

The black reference curve is static because the PDE is stationary; colored curves show DTB at selected outer iterations.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for axis, (d, result) in zip(axes, results.items()):
    t = result['diagonal']
    reference = np.cos(0.5 * np.pi * t) ** d
    axis.plot(t, reference, 'k--', lw=2, label='reference')
    for step, values in result['snapshots'].items():
        axis.plot(t, values, label=f'DTB step {step}')
    axis.set(title=f'd={d}', xlabel='diagonal coordinate', ylabel='u')
    axis.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Only the four requested diagnostics.
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for d, result in results.items():
    h = result['history']; steps = [r['step'] for r in h]
    axes[0,0].semilogy(steps, [r['relative_l2'] for r in h], label=f'd={d}')
    axes[0,1].plot(steps, [r['F_theta'] for r in h], label=f'd={d}')
    axes[1,0].semilogy(steps, [r['inner_residual'] for r in h], label=f'd={d}')
    axes[1,1].semilogy(steps, [r['alpha_norm'] for r in h], label=f'd={d}')
for axis, title in zip(axes.flat, ['relative L2 error', 'F(theta)', 'inner residual', '||alpha||2']):
    axis.set(xlabel='outer iteration', title=title); axis.grid(alpha=.25); axis.legend()
plt.tight_layout(); plt.show()